<a href="https://colab.research.google.com/github/ArjunBhakta/Data-Science-Cohort-20/blob/main/Project_5_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


#Part 1)



- The CSV file is available at https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv
- The file contains a list of famous people and a brief overview.
- The goal of part 1) is to ...
  1. Pick one person from the list ( the reference person ) and output 10 other people who's overview are "closest" to the reference person in a Natural Language Processing sense
  1. Also output the sentiment of the overview of the reference person



## Import Packages

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
!pip install textblob -q
from textblob import TextBlob
!pip install umap-learn -q
from sklearn.decomposition import TruncatedSVD
import umap
import plotly.express as px

## Functions

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from textblob import TextBlob
import pandas as pd


def get_sentiment(text):
    blob = TextBlob(str(text))

    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    if polarity > 0.1:
        sentiment = "Positive"
    elif polarity < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return pd.Series({
        "polarity": round(polarity, 3),
        "subjectivity": round(subjectivity, 3),
        "sentiment": sentiment
    })


def find_similar_people(
    reference_person,
    df,
    tfidf_matrix,
    top_n=10,
    include_sentiment=True
):
    """
    Find most similar people using cosine similarity and compare
    against Euclidean distance rankings.
    """

    ref_idx = df[df["name"] == reference_person].index

    if len(ref_idx) == 0:
        print(f"'{reference_person}' not found.")
        return None

    ref_idx = ref_idx[0]

    cosine_scores = cosine_similarity(
        tfidf_matrix[ref_idx],
        tfidf_matrix
    ).flatten()

    euclidean_scores = euclidean_distances(
        tfidf_matrix[ref_idx],
        tfidf_matrix
    ).flatten()

    cosine_ranked = [
        i for i in cosine_scores.argsort()[::-1]
        if i != ref_idx
    ]

    euclidean_ranked = [
        i for i in euclidean_scores.argsort()
        if i != ref_idx
    ]

    top_indices = cosine_ranked[:top_n]

    results = df.iloc[top_indices][["name", "decodedName"]].copy()

    results["cosine_similarity"] = (
        cosine_scores[top_indices].round(4)
    )

    results["euclidean_distance"] = (
        euclidean_scores[top_indices].round(4)
    )

    cosine_rank_lookup = {
        idx: rank + 1
        for rank, idx in enumerate(cosine_ranked)
    }

    euclidean_rank_lookup = {
        idx: rank + 1
        for rank, idx in enumerate(euclidean_ranked)
    }

    results["cosine_rank"] = [
        cosine_rank_lookup[idx]
        for idx in top_indices
    ]

    results["euclidean_rank"] = [
        euclidean_rank_lookup[idx]
        for idx in top_indices
    ]

    results["rank_difference"] = (
        results["cosine_rank"]
        - results["euclidean_rank"]
    ).abs()

    if include_sentiment:
        sentiment_df = (
            df.iloc[top_indices]["text"]
            .apply(get_sentiment)
        )

        results = pd.concat(
            [results, sentiment_df],
            axis=1
        )

    results = results.sort_values(
        "cosine_rank"
    ).reset_index(drop=True)

    results.index += 1

    return results

### Compute umap ahead of time to save time

In [ ]:
def compute_umap_projection(
    df,
    tfidf_matrix,
    n_svd_components=100,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
):
    svd = TruncatedSVD(
        n_components=n_svd_components,
        random_state=random_state
    )

    coords_svd = svd.fit_transform(tfidf_matrix)

    print(f"SVD output shape: {coords_svd.shape}")
    print(f"Variance explained: {svd.explained_variance_ratio_.sum():.2%}")

    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="cosine",
        random_state=random_state
    )

    coords_umap = reducer.fit_transform(coords_svd)

    plot_df = pd.DataFrame({
        "x": coords_umap[:, 0],
        "y": coords_umap[:, 1],
        "z": coords_umap[:, 2],
        "name": df["name"].values,
        "decodedName": df["decodedName"].values,
    })

    return plot_df, svd, reducer

### Plotting function against umap

In [ ]:
def plot_similar_people_umap(
    plot_df_umap,
    reference_person,
    ref_idx,
    similar_indices
):
    plot_df = plot_df_umap.copy()

    plot_df["highlight"] = "Other"
    plot_df.loc[similar_indices, "highlight"] = "Top 10 Similar"
    plot_df.loc[ref_idx, "highlight"] = "Reference"

    fig = px.scatter_3d(
        plot_df,
        x="x",
        y="y",
        z="z",
        color="highlight",
        hover_name="decodedName",
        hover_data={
            "name": True,
            "x": False,
            "y": False,
            "z": False,
        },
        color_discrete_map={
            "Reference": "red",
            "Top 10 Similar": "orange",
            "Other": "lightblue",
        },
        title=f"UMAP 3D — {reference_person} and Top 10 Similar People",
        opacity=0.5,
    )

    fig.update_traces(marker_size=3)
    fig.update_traces(marker_size=10, selector={"name": "Top 10 Similar"})
    fig.update_traces(
        marker_size=10,
        marker_symbol="diamond",
        selector={"name": "Reference"}
    )

    fig.show()

In [ ]:
def get_similarity_results(reference_person, df, tfidf_matrix, top_n=10):
    ref_matches = df[df["name"] == reference_person].index

    if len(ref_matches) == 0:
        print(f"'{reference_person}' not found.")
        return None, None, None

    ref_idx = ref_matches[0]

    cosine_scores = cosine_similarity(
        tfidf_matrix[ref_idx],
        tfidf_matrix
    ).flatten()

    similar_indices = cosine_scores.argsort()[::-1]
    similar_indices = [i for i in similar_indices if i != ref_idx][:top_n]

    results = df.iloc[similar_indices][["name", "decodedName"]].copy()
    results["cosine_similarity"] = cosine_scores[similar_indices].round(4)

    results = results.reset_index(drop=True)
    results.index += 1

    return results, ref_idx, similar_indices

In [ ]:
def print_bios(results, df, max_chars=None):

    """

    Print biographies for people in a similarity results dataframe.

    Parameters

    ----------

    results : DataFrame

        Output from find_similar_people()

    df : DataFrame

        Original dataframe containing name and text columns

    max_chars : int or None

        Truncate bios for easier reading

    """

    for rank, row in results.iterrows():

        person = row["name"]

        display_name = row["decodedName"]

        bio = df.loc[

            df["name"] == person,

            "text"

        ].iloc[0]

        if max_chars is not None:

            bio = bio[:max_chars] + "..."

        print("=" * 80)

        print(f"Rank {rank + 1}: {display_name}")

        print("=" * 80)

        print(bio)

        print("\n")

## Read in Data

In [ ]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'
df = pd.read_csv(url)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42786 entries, 0 to 42785
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   URI     42786 non-null  object
 1   name    42786 non-null  object
 2   text    42786 non-null  object
dtypes: object(3)
memory usage: 1002.9+ KB


In [ ]:
sample  = df.sample()
sample

,URI,name,text
25491,<http://dbpedia.org/resource/Brush_Shiels>,Brush Shiels,brendan ian brush shiels born in 1952 dublin i...


In [ ]:
reference_person = 'Tim Bartro'

## Add column with decoded names ( safe url encoding was used in dbpedia )

In [ ]:
from urllib.parse import unquote

decoded = df["name"].apply(unquote)
num_encoded = (df["name"] != decoded).sum()
print(f"{num_encoded:,} rows contain URL-encoded characters")

2,485 rows contain URL-encoded characters


In [ ]:
df["decodedName"] = df["name"].apply(unquote)

In [ ]:
changed = df[df["name"] != df["decodedName"]]
changed[["name", "decodedName"]].head(20)

,name,decodedName
35,Freimut B%C3%B6rngen,Freimut Börngen
36,Th%C3%BCring Br%C3%A4m,Thüring Bräm
73,Marcel J. Melan%C3%A7on,Marcel J. Melançon
85,Zvonimir Juri%C4%87,Zvonimir Jurić
122,Se%C3%A1n %C3%93g %C3%93 hAilp%C3%ADn,Seán Óg Ó hAilpín
129,L%C3%A1zaro C%C3%A1rdenas Batel,Lázaro Cárdenas Batel
148,Ren%C3%A9 Froger,René Froger
166,Ant%C3%B3nio Castanheira Neves,António Castanheira Neves
182,Mike P%C3%A9rez (baseball),Mike Pérez (baseball)
203,David Pastr%C5%88%C3%A1k,David Pastrňák


In [ ]:
df.sample()

,URI,name,text,decodedName
9129,<http://dbpedia.org/resource/Jan_Tore_Odegard>,Jan Tore Odegard,jan tore odegard born 14 october 1940 is a nor...,Jan Tore Odegard


In [ ]:
changed.shape

(2485, 4)

## Setup TF-IDF Model on short Bio Info

In [ ]:
# Vectorize the words with sklearn TFIDVectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(df['text'])
print(f'Matrix shape: {tfidf_matrix.shape}')  # (people, unique words)


Matrix shape: (42786, 10000)


In [ ]:
# Change this to any name from data frame
reference_person = "Bruce Taylor (baseball)"

In [ ]:
results = find_similar_people(
    reference_person=reference_person,
    df=df,
    tfidf_matrix=tfidf_matrix,
    top_n=10

)

results

,name,decodedName,cosine_similarity,euclidean_distance,cosine_rank,euclidean_rank,rank_difference,polarity,subjectivity,sentiment
1,Don Secrist,Don Secrist,0.3977,1.0976,1,1,0,0.080,0.313,Neutral
2,Jim Stump,Jim Stump,0.3948,1.1002,2,2,0,0.126,0.365,Positive
3,Vern Geishert,Vern Geishert,0.3824,1.1114,3,3,0,0.013,0.325,Neutral
4,John Tsitouris,John Tsitouris,0.3754,1.1177,4,4,0,0.044,0.382,Neutral
5,Jamie Brewington,Jamie Brewington,0.3740,1.1189,5,5,0,0.040,0.418,Neutral
6,Randor Bierd,Randor Bierd,0.3718,1.1209,6,6,0,0.098,0.287,Neutral
7,Rick Grapenthin,Rick Grapenthin,0.3687,1.1237,7,7,0,0.059,0.334,Neutral
8,Tom Flanigan (baseball),Tom Flanigan (baseball),0.3670,1.1252,8,8,0,0.013,0.228,Neutral
9,Jim Duffalo,Jim Duffalo,0.3628,1.1289,9,9,0,0.082,0.289,Neutral
10,Steve Kline (right-handed pitcher),Steve Kline (right-handed pitcher),0.3577,1.1334,10,10,0,0.067,0.260,Neutral


In [ ]:
print_bios(results, df)

Rank 2: Don Secrist
donald laverne secrist born february 26 1944 is an american former professional baseball player a lefthanded pitcher who appeared in 28 games played all in relief for the 19691970 chicago white sox of major league baseball he stood 6 feet 2 inches 188 m tall and weighed 195 pounds 88 kgsecrist had two outstanding seasons in minor league baseball after signing with the baltimore orioles he was undefeated in seven decisions with a 196 earned run average for the 1963 aberdeen pheasants of the class a northern league drafted from the orioles by the cincinnati reds following that season secrist spent five more years in the reds farm system in his last in 1968 he won 11 games and lost two for the indianapolis indians of the triplea pacific coast league following that campaign he was dealt with catcher don pavletich to the white sox for pitcher jack fishersecrist then spent much of the 1969 season with the mlb white sox appearing in 19 games making his debut during the hom

In [ ]:
plot_df_umap, svd_model, umap_model = compute_umap_projection(

    df,

    tfidf_matrix

)

SVD output shape: (42786, 100)
Variance explained: 16.63%


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:
reference_person = "Bruce Taylor (baseball)"
results, ref_idx, similar_indices = get_similarity_results(

    reference_person,
    df,
    tfidf_matrix,
    top_n=10

)

print(results)

plot_similar_people_umap(

    plot_df_umap,
    reference_person,
    ref_idx,
    similar_indices

)

                                  name                         decodedName  \
1                          Don Secrist                         Don Secrist   
2                            Jim Stump                           Jim Stump   
3                        Vern Geishert                       Vern Geishert   
4                       John Tsitouris                      John Tsitouris   
5                     Jamie Brewington                    Jamie Brewington   
6                         Randor Bierd                        Randor Bierd   
7                      Rick Grapenthin                     Rick Grapenthin   
8              Tom Flanigan (baseball)             Tom Flanigan (baseball)   
9                          Jim Duffalo                         Jim Duffalo   
10  Steve Kline (right-handed pitcher)  Steve Kline (right-handed pitcher)   

    cosine_similarity  
1              0.3977  
2              0.3948  
3              0.3824  
4              0.3754  
5              0.3740

## Issues/Observation - Some people are matched on country of origin rather than occupation. This may not be the bias we want

In [ ]:
reference_person_1 = "Iradj Fazel"

results_1, ref_idx_1, similar_indices_1 = get_similarity_results(

    reference_person_1,
    df,
    tfidf_matrix,
    top_n=10

)


display(results_1)

plot_similar_people_umap(

    plot_df_umap,
    reference_person,
    ref_idx_1,
    similar_indices_1

)

,name,decodedName,cosine_similarity
1,Maziar Ashrafian Bonab,Maziar Ashrafian Bonab,0.4630
2,Sima Bina,Sima Bina,0.4437
3,Mohammad Reza Azadehfar,Mohammad Reza Azadehfar,0.4104
4,Moslem Eskandar-Filabi,Moslem Eskandar-Filabi,0.4103
5,Maryam Akhondy,Maryam Akhondy,0.4032
6,Sadegh Kharazi,Sadegh Kharazi,0.4028
7,Mahsa Vahdat,Mahsa Vahdat,0.3673
8,Hossein Marashi,Hossein Marashi,0.3639
9,Hassan Shariatmadari,Hassan Shariatmadari,0.3559
10,Firoozeh Dumas,Firoozeh Dumas,0.3557


In [ ]:
print_bios(results_1,df)

Rank 2: Maziar Ashrafian Bonab
dr maziar ashrafian bonab persian is an iranian forensic and medical geneticistspecialising in forensic genetics the use of the dna markers in the investigation of crimes and ancestry and forensic facial reconstruction his groundbreaking research uses human dna markers mainly mtdna and the y chromosome markers to identify the ancestral history of humanshuman populations in both anthropological and forensic cases his main area of research is the population history of the middle east specifically iranmaziar was born in tehran iran 20 september 1966 before completing his phd in cambridge he first qualified as a medical doctor from tehran university of medical sciences 19841991 and worked as a medical practitioner in iran after completing a postgraduate course in forensic medicine at the iranian legal medical organization 1992 he worked as the head of hormozgan province legal medical centre iran for four years 19921996 as well as dealing with many different f

# Part 2)



- For the same reference person that you chose in Part 1), use the Wikipedia API to access the whole content of the reference person's Wikipedia page.
- The goal of Part 2) is to ...
  1. Print out the text of the Wikipedia article for the reference person
  1. Determine the sentiment of the text of the Wikipedia page for the reference person
  1. Collect the text of the Wikipedia pages from the 10 nearest neighbors from Part 1)
  1. Determine the nearness ranking of these 10 people to your reference person based on their entire Wikipedia page
  1. Compare, i.e. plot,  the nearest ranking from Step 1) with the Wikipedia page nearness ranking.  A difference of the rank is one means of comparison.



In [ ]:
%%capture
!pip3 install wikipedia-api
import wikipediaapi

wiki = wikipediaapi.Wikipedia(language='en', user_agent='Project5/1.0')

def get_wiki_text(name):
    page = wiki.page(name)
    if page.exists():
        return page.text
    return ""

ref_wiki_text = get_wiki_text(reference_person)
print(f"Wikipedia article length: {len(ref_wiki_text.split())} words\n")
print(ref_wiki_text[:3000])


In [ ]:
from textblob import TextBlob

blob = TextBlob(ref_wiki_text)
polarity     = blob.sentiment.polarity
subjectivity = blob.sentiment.subjectivity

label = 'Positive' if polarity > 0.1 else ('Negative' if polarity < -0.1 else 'Neutral')

print(f"Wikipedia Sentiment for: {reference_person}")
print(f"  Polarity:     {polarity:.3f}  ({label})")
print(f"  Subjectivity: {subjectivity:.3f}  ({'Subjective' if subjectivity > 0.5 else 'Objective'})")


Wikipedia Sentiment for: Bruce Taylor (baseball)
  Polarity:     0.122  (Positive)
  Subjectivity: 0.474  (Objective)


In [ ]:
neighbor_names = df.iloc[similar_indices]['name'].tolist()

wiki_texts = {}
for name in neighbor_names:
    text = get_wiki_text(name)
    wiki_texts[name] = text
    status = f"{len(text.split())} words" if text else "NOT FOUND"
    print(f"  {name}: {status}")


  Don Secrist: 358 words
  Jim Stump: 320 words
  Vern Geishert: 260 words
  John Tsitouris: 475 words
  Jamie Brewington: 477 words
  Randor Bierd: 308 words
  Rick Grapenthin: 489 words
  Tom Flanigan (baseball): 265 words
  Jim Duffalo: 391 words
  Steve Kline (right-handed pitcher): 303 words


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus_names = [reference_person] + neighbor_names
corpus_texts = [ref_wiki_text] + [wiki_texts[n] for n in neighbor_names]

wiki_vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
wiki_matrix = wiki_vectorizer.fit_transform(corpus_texts)

wiki_sims = cosine_similarity(wiki_matrix[0], wiki_matrix[1:]).flatten()

comparison_df = pd.DataFrame({
    'name': neighbor_names,
    'dbpedia_rank': range(1, 11),
    'wiki_similarity': wiki_sims,
})

comparison_df['wiki_rank'] = comparison_df['wiki_similarity'].rank(ascending=False).astype(int)
comparison_df['rank_change'] = comparison_df['dbpedia_rank'] - comparison_df['wiki_rank']
print(comparison_df.to_string(index=False))


                              name  dbpedia_rank  wiki_similarity  wiki_rank  rank_change
                       Don Secrist             1         0.163162          4           -3
                         Jim Stump             2         0.182715          3           -1
                     Vern Geishert             3         0.119482         10           -7
                    John Tsitouris             4         0.140681          8           -4
                  Jamie Brewington             5         0.153645          7           -2
                      Randor Bierd             6         0.131788          9           -3
                   Rick Grapenthin             7         0.156597          6            1
           Tom Flanigan (baseball)             8         0.160185          5            3
                       Jim Duffalo             9         0.193802          2            7
Steve Kline (right-handed pitcher)            10         0.231585          1            9


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=comparison_df['name'], y=comparison_df['dbpedia_rank'],
    mode='lines+markers', name='DBpedia Rank',
    marker=dict(size=10, color='steelblue'),
))

fig.add_trace(go.Scatter(
    x=comparison_df['name'], y=comparison_df['wiki_rank'],
    mode='lines+markers', name='Wikipedia Rank',
    marker=dict(size=10, color='coral'),
))

fig.update_layout(
    title=f'DBpedia vs Wikipedia Ranking — relative to {reference_person}',
    xaxis_title='Person',
    yaxis_title='Rank (1 = most similar)',
    yaxis=dict(autorange='reversed'),
    xaxis_tickangle=45,
    height=500,
)
fig.show()


# Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

name_options = sorted(df["decodedName"].dropna().unique().tolist())

person_picker = widgets.Combobox(
    placeholder="Type a famous person...",
    options=name_options,
    description="Person:",
    #ensure_option=True,
    layout=widgets.Layout(width="500px")
)

def set_reference_person(decoded_name):
    global reference_person

    if decoded_name:
        reference_person = df.loc[
            df["decodedName"] == decoded_name,
            "name"
        ].iloc[0]

        print(f"reference_person set to: {reference_person}")

widgets.interact(
    set_reference_person,
    decoded_name=person_picker
);



interactive(children=(Combobox(value='', description='Person:', layout=Layout(width='500px'), options=(' Renat…

In [ ]:
results, ref_idx, similar_indices = get_similarity_results(

    reference_person,

    df,

    tfidf_matrix,

    top_n=10

)

display(results)
print_bios(results, df)

plot_similar_people_umap(

    plot_df_umap,
    reference_person,
    ref_idx,
    similar_indices

)

,name,decodedName,cosine_similarity
1,Don Secrist,Don Secrist,0.3977
2,Jim Stump,Jim Stump,0.3948
3,Vern Geishert,Vern Geishert,0.3824
4,John Tsitouris,John Tsitouris,0.3754
5,Jamie Brewington,Jamie Brewington,0.3740
6,Randor Bierd,Randor Bierd,0.3718
7,Rick Grapenthin,Rick Grapenthin,0.3687
8,Tom Flanigan (baseball),Tom Flanigan (baseball),0.3670
9,Jim Duffalo,Jim Duffalo,0.3628
10,Steve Kline (right-handed pitcher),Steve Kline (right-handed pitcher),0.3577


Rank 2: Don Secrist
donald laverne secrist born february 26 1944 is an american former professional baseball player a lefthanded pitcher who appeared in 28 games played all in relief for the 19691970 chicago white sox of major league baseball he stood 6 feet 2 inches 188 m tall and weighed 195 pounds 88 kgsecrist had two outstanding seasons in minor league baseball after signing with the baltimore orioles he was undefeated in seven decisions with a 196 earned run average for the 1963 aberdeen pheasants of the class a northern league drafted from the orioles by the cincinnati reds following that season secrist spent five more years in the reds farm system in his last in 1968 he won 11 games and lost two for the indianapolis indians of the triplea pacific coast league following that campaign he was dealt with catcher don pavletich to the white sox for pitcher jack fishersecrist then spent much of the 1969 season with the mlb white sox appearing in 19 games making his debut during the hom